<a href="https://colab.research.google.com/github/saichandanss2190/ITA0615-Machine-Learning-192312190-/blob/main/Exp_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. The ID3 Decision Tree Algorithm

The ID3 (Iterative Dichotomiser 3) algorithm builds a decision tree by employing a top-down, greedy search through the space of possible branches. It uses **Entropy** and **Information Gain** to select the attribute that best splits the training data at each node.

In [6]:
import numpy as np
import pandas as pd

# Let's define the PlayTennis dataset
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'Play': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
}

df = pd.DataFrame(data)
display(df)

,Outlook,Temperature,Humidity,Wind,Play
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


### 2. ID3 Implementation Helper Functions
We will define functions to calculate standard entropy, average attribute entropy, and the resulting Information Gain.

In [7]:
def calculate_entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)
    entropy = 0
    for i in range(len(elements)):
        probability = counts[i] / np.sum(counts)
        entropy -= probability * np.log2(probability)
    return entropy

def calculate_info_gain(data, split_attribute_name, target_name="Play"):
    # Calculate total entropy of the target
    total_entropy = calculate_entropy(data[target_name])

    # Calculate the values and corresponding counts for the splitting attribute
    vals, counts= np.unique(data[split_attribute_name], return_counts=True)

    # Calculate the weighted entropy of the attribute
    weighted_entropy = 0
    for i in range(len(vals)):
        subset = data[data[split_attribute_name] == vals[i]]
        subset_entropy = calculate_entropy(subset[target_name])
        weighted_entropy += (counts[i] / np.sum(counts)) * subset_entropy

    # Information Gain is the difference
    information_gain = total_entropy - weighted_entropy
    return information_gain

### 3. Recursive ID3 Tree Building Function

In [8]:
def id3(data, original_data, features, target_attribute_name="Play", parent_node_class=None):
    # 1. If all target values are the same, return this value
    if len(np.unique(data[target_attribute_name])) <= 1:
        return np.unique(data[target_attribute_name])[0]

    # 2. If dataset is empty, return the mode target value in the original dataset
    elif len(data) == 0:
        return np.unique(original_data[target_attribute_name])[np.argmax(np.unique(original_data[target_attribute_name], return_counts=True)[1])]

    # 3. If features list is empty, return parent node class
    elif len(features) == 0:
        return parent_node_class

    else:
        # Set the default value for this node to the majority class
        parent_node_class = np.unique(data[target_attribute_name])[np.argmax(np.unique(data[target_attribute_name], return_counts=True)[1])]

        # Select the feature which best splits the dataset
        item_values = [calculate_info_gain(data, feature, target_attribute_name) for feature in features]
        best_feature_index = np.argmax(item_values)
        best_feature = features[best_feature_index]

        # Create the tree structure
        tree = {best_feature: {}}

        # Remove the feature that has been selected
        remaining_features = [i for i in features if i != best_feature]

        # Grow a branch for each value of the selected feature
        for value in np.unique(data[best_feature]):
            sub_data = data[data[best_feature] == value]

            # Recurse to build the subtree
            subtree = id3(sub_data, data, remaining_features, target_attribute_name, parent_node_class)
            tree[best_feature][value] = subtree

        return tree

### 4. Training the ID3 Decision Tree
Let's build the decision tree using our dataset.

In [9]:
features = [col for col in df.columns if col != 'Play']
tree = id3(df, df, features)

import pprint
print("Built ID3 Decision Tree Structure:")
pprint.pprint(tree)

Built ID3 Decision Tree Structure:
{'Outlook': {'Overcast': 'Yes',
             'Rain': {'Wind': {'Strong': 'No', 'Weak': 'Yes'}},
             'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}}}


### 5. Classification and Prediction
We'll create a prediction function to classify new unseen samples using our built tree.

In [10]:
def predict(query, tree, default='Yes'):
    for key in list(query.keys()):
        if key in list(tree.keys()):
            try:
                result = tree[key][query[key]]
            except KeyError:
                return default

            if isinstance(result, dict):
                return predict(query, result)
            else:
                return result

# Define a new sample to classify
new_sample = {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}

prediction = predict(new_sample, tree)
print(f"New sample attributes: {new_sample}")
print(f"Predicted target class (Play): {prediction}")

New sample attributes: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted target class (Play): No
